# F3 — Spike Comparison W3 vs W4

In [ ]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)

# SEAGATE required for E notebooks


In [ ]:
def load_spike_cal(wm, day, t_center=pd.Timestamp('14:00:00').time()):
    mids = {}
    for sym in [wm['front'], wm['back']]:
        fpath = list(SEAGATE_DIR.glob(f'mbp10_*{sym}*{day}*.parquet'))
        if not fpath: return None
        df = pd.read_parquet(fpath[0], columns=['ts_event','bid_px_00','ask_px_00'])
        df['ts_event'] = pd.to_datetime(df['ts_event'])
        mask = (df['ts_event'].dt.hour == 13) & (df['ts_event'].dt.minute >= 55) |                (df['ts_event'].dt.hour == 14) & (df['ts_event'].dt.minute <= 5)
        df = df[mask].set_index('ts_event')
        mids[sym] = ((df['bid_px_00']+df['ask_px_00'])/2).resample('1ms').last().ffill()
        del df; gc.collect()
    if len(mids) < 2: return None
    cal = mids[wm['back']].subtract(mids[wm['front']]).dropna()
    return cal

w3_cal = load_spike_cal(WINDOWS_META['W3'], '2025-03-14')
w4_cal = load_spike_cal(WINDOWS_META['W4'], '2025-06-13')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, cal, wk, day in [(axes[0],w3_cal,'W3','2025-03-14'),(axes[1],w4_cal,'W4','2025-06-13')]:
    if cal is None:
        ax.text(0.5,0.5,'Data missing',ha='center',transform=ax.transAxes); continue
    t0 = pd.Timestamp(f'{day} 14:00:00')
    secs = (cal.index - t0).total_seconds()
    ax.plot(secs, cal.values, lw=0.8, color=WIN_COLORS[wk])
    ax.axvline(0, color='red', ls='--', lw=1.2, label='t=14:00:00')
    ax.set_xlim(-300, 300)
    ax.set_xlabel('Seconds relative to 14:00 UTC')
    ax.set_ylabel('Calendar Spread (pts)')
    ax.set_title(f'{wk}: {day}')
    ax.legend()
fig.suptitle('Flash Spike Comparison: W3 Mar-14 vs W4 Jun-13 (±5 min)', fontsize=13)
save_fig(fig, 'F3_spike_comparison.png')
